In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot

from model.dataset import BagsDataset, custom_collate_fn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
adata = sc.read_h5ad('../example.h5ad')
binding_affinity = pd.read_csv('../binding_affinity.csv', index_col=0)

In [4]:
dataset = BagsDataset(adata, binding_affinity, radius= 200)

Checking data for bag 0:
Number of cells in this bag: 9
Sample of circle_barcodes: ['AAACAAGTATCTCCCA-1' 'ACTAGTTGCGATCGTC-1' 'AGAGGCTTCGGAAACC-1'
 'AGTTTGGCACGGGTTG-1' 'CAGCAGTCCAGACTAT-1']
Shape of affinity_data: (9, 19379)
Bag 0 has 9 instances
Bag 1 has 9 instances
Bag 2 has 9 instances
Bag 3 has 7 instances
Bag 4 has 5 instances
Bag 5 has 9 instances
Bag 6 has 5 instances
Bag 7 has 9 instances
Bag 8 has 8 instances
Bag 9 has 9 instances
Bag 10 has 9 instances
Bag 11 has 9 instances
Bag 12 has 4 instances
Bag 13 has 9 instances
Bag 14 has 9 instances
Bag 15 has 8 instances
Bag 16 has 8 instances
Bag 17 has 9 instances
Bag 18 has 9 instances
Bag 19 has 9 instances
Bag 20 has 9 instances
Bag 21 has 8 instances
Bag 22 has 8 instances
Bag 23 has 9 instances
Bag 24 has 9 instances
Bag 25 has 9 instances
Bag 26 has 9 instances
Bag 27 has 9 instances
Bag 28 has 9 instances
Bag 29 has 9 instances
Bag 30 has 9 instances
Bag 31 has 9 instances
Bag 32 has 9 instances
Bag 33 has 9 instances
Ba

In [5]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

In [6]:
distances, gene_expressions, affinity_data, labels = next(iter(dataloader))

In [7]:
class Sparsemax(nn.Module):
    def __init__(self, dim=None):
        super(Sparsemax, self).__init__()
        self.dim = -1 if dim is None else dim

    def forward(self, input):
        input = input.transpose(0, self.dim)
        original_size = input.size()
        input = input.reshape(input.size(0), -1)
        input = input.transpose(0, 1)
        dim = 1

        number_of_logits = input.size(dim)

        input = input - torch.max(input, dim=dim, keepdim=True)[0].expand_as(input)

        zs = torch.sort(input=input, dim=dim, descending=True)[0]
        range = torch.arange(start=1, end=number_of_logits + 1, step=1, device=input.device, dtype=input.dtype).view(1, -1)
        range = range.expand_as(zs)

        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(input.type())
        k = torch.max(is_gt * range, dim, keepdim=True)[0]

        zs_sparse = is_gt * zs

        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(input)

        output = F.relu(input - taus)

        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)

        return output

In [8]:
model = Sparsemax()
output = model(gene_expressions[0])
output

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [9]:
print(torch.count_nonzero(gene_expressions[0] == 0)) 
print(torch.count_nonzero(output == 0)) 

tensor(114868)
tensor(135646)


In [10]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0))
        self.sparsemax = Sparsemax(dim=0)
    
    def forward(self, x):
        x = self.sparsemax(x)
        #print(x)
        a = torch.sigmoid(self.a)  
        x = torch.exp(-a * x) 
        return x

In [11]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[0.8330],
        [0.8330],
        [1.0000],
        [1.0000],
        [0.8330],
        [0.8330],
        [1.0000]], grad_fn=<ExpBackward0>)


In [12]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0))
        self.sparsemax = Sparsemax(dim=-1) 

    def forward(self, x):
        x = self.sparsemax(x)
        b = torch.sigmoid(self.b)
        x = torch.exp(-b * x)
        return x


In [13]:
gene_expressions[0].shape

torch.Size([7, 19379])

In [14]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output.shape)

torch.Size([7, 19379])


In [15]:
class Affinity(nn.Module):
    def __init__(self):
        super(Affinity, self).__init__()
        self.c = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, x):
        x = F.softmax(x, dim=-1)
        c = torch.sigmoid(self.c)
        x = torch.exp(-c * x)
        return x

In [16]:
model = Affinity()
input_tensor = affinity_data[0]
output = model(input_tensor)
output

tensor([[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        ...,
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000]],
       grad_fn=<ExpBackward0>)

In [17]:
print(output.shape)

torch.Size([7, 19379])


In [18]:
class MILPooling(nn.Module):
    def __init__(self):
        super(MILPooling, self).__init__()
        self.pool = lambda x: torch.sum(x, dim=0)
    
    def forward(self, x):
        x = self.pool(x)
        #print(x)
        return x
        

In [19]:
class MIL(nn.Module):
    def __init__(self):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.affinity = Affinity()
        self.pooling = MILPooling()
    
    def forward(self, distances, gene_expressions, affinity_data):
        instance_outputs = []
        for distance, gene_expression, affinity in zip(distances, gene_expressions, affinity_data):
            distance = self.distance(distance)
            #print(distance.shape)
            gene_expression = self.gene_expression(gene_expression)
            #print(gene_expression[0])
            #print(gene_expression.shape)
            affinity = self.affinity(affinity)
            #print(affinity,shape)
            
            combined_output = distance * gene_expression * affinity
            #combined_output = combined_output.unsqueeze(0)
            #print(combined_output.dim())
            pooled_output = self.pooling(combined_output)
            
            output = torch.sigmoid(pooled_output)
            instance_outputs.append(output)
            #print(output)

        bag_output = torch.sum(torch.stack(instance_outputs), dim=0)
        
        bag_output = torch.sigmoid(bag_output)
        return bag_output


In [59]:
class MIL(nn.Module):
    def __init__(self):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.affinity = Affinity()
        #self.pooling = MILPooling()
    
    def forward(self, distances, gene_expressions, affinity_data):
        instance_outputs = []
        z = []
        for distance, gene_expression, affinity in zip(distances, gene_expressions, affinity_data):
            distance = self.distance(distance)
            #print(distance.shape)
            gene_expression = self.gene_expression(gene_expression)
            #print(gene_expression.shape)
            affinity = self.affinity(affinity)
            #print(affinity,shape)
            for j in range(len(gene_expression)):
                zj = gene_expression[j, :] * affinity[j, :] 
                z.append(zj)
            #print(z)
            #print("***")
            z = torch.sum(torch.stack(z), dim=1)
            #print(z.shape)
            #print("***")
            #print(distance.squeeze().shape)

            distance = distance.squeeze()

            instance_output = distance * z
            instance_outputs.append(instance_output)
        #print(instance_outputs)

        bag_output = torch.sum(torch.stack(instance_outputs), dim=1)
        #print(bag_output.shape)

        bag_output = torch.sigmoid(bag_output)
        print(bag_output)
        
        return bag_output
    




In [60]:
model = MIL()
output = model(distances, gene_expressions, affinity_data)

tensor([1.], grad_fn=<SigmoidBackward0>)


In [64]:
output

tensor([1.], grad_fn=<SigmoidBackward0>)

In [65]:
labels[0]


tensor(0.)

In [63]:
make_dot(output, params=dict(model.named_parameters())).render("MIL_computational_graph", format="png")


'MIL_computational_graph.png'

In [55]:
print(model)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (sparsemax): Sparsemax()
  )
  (affinity): Affinity()
)


In [56]:
model = MIL()
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (sparsemax): Sparsemax()
  )
  (affinity): Affinity()
)

In [67]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    for i, (distances, gene_expressions, affinity_data, label) in enumerate(dataloader):

        optimizer.zero_grad()
        print("*******label")
        print(label)
        output = model(distances, gene_expressions, affinity_data)
        print("*******output")
        print(output)
        loss = criterion(output, label)
    
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')


*******label
tensor([0.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([0.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([1.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([1.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([1.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([1.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([0.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([0.])
tensor([1.], grad_fn=<SigmoidBackward0>)
*******output
tensor([1.], grad_fn=<SigmoidBackward0>)
*******label
tensor([0.]

KeyboardInterrupt: 